# 5. Conclusiones y limitaciones

In [45]:
display(Markdown(f"""
### 5.1 Conclusiones

1. **Base de datos.** *Bank Marketing* (UCI, CC BY 4.0) cumple el tamaño exigido ({num(n_filas)} registros, {p_pred} predictoras) y plantea un problema de clasificación desbalanceado ({pct(n_pos / n_filas)} de positivos). Su calidad es buena en forma ({'sin errores de escritura ni nulos explícitos' if sin_errores_cat and df.isna().sum().sum() == 0 else 'con algunos problemas de formato'}), pero tiene faltantes codificados como `unknown` en el {pct(alguno.mean())} de los registros, que no son MCAR y son informativos, {n_dup_exactos} duplicados exactos y un código especial (`pdays = 999`) inconsistente con `previous` en {num(inc_pdays)} registros.
2. **Partición.** Las filas están ordenadas en el tiempo, así que la partición es cronológica: el entrenamiento va de {periodo_tr.iloc[0]} a {periodo_tr.iloc[-1]} y la prueba de {periodo_te.iloc[0]} a {periodo_te.iloc[-1]}. La prevalencia pasa del {pct(y_tr.mean())} en entrenamiento al {pct(prev_te)} en prueba, un cambio que el modelo no puede observar durante el ajuste.
3. **Fuga.** `duration` es una fuga evidente: se conoce solo al terminar la llamada y sube el AUC de validación temporal de {num(auc_sin_dur, 3)} a {num(auc_con_dur, 3)}. Se excluyó. Dentro de entrenamiento, una validación aleatoria habría dado AUC de {num(auc_alea, 3)} con el instrumento de auditoría y un modelo que solo conoce el periodo, {num(auc_periodo, 3)}, frente a {num(cv_tss['AUC'].mean(), 3)} con validación temporal: esa **fuga temporal** justifica la partición y la validación cronológicas.
4. **EDA.** Sin `duration`, la mayor información mutua con `y` es la de {enumerar([c for c in asoc.index if c != 'duration'][:4])}, que describen el contexto del periodo; el historial con el banco y el canal aportan menos, y los atributos del cliente, todavía menos. Las macro son redundantes entre sí (VIF inicial de hasta {num(vif_inicial.max(), 1)}), por lo que se retiraron {enumerar(NUM_RETIRADAS_VIF)}. Las subpoblaciones encontradas corresponden a regímenes económicos y al historial de contacto, no a perfiles sociodemográficos.
5. **Modelo base.** La regresión logística (C = {num_c(C_opt)}) logra en prueba AUC de {num(P[nombre_u]['AUC'], 3)} [{num(IC[nombre_u]['AUC'][0], 3)}; {num(IC[nombre_u]['AUC'][1], 3)}] y AUC-PR de {num(P[nombre_u]['AUC-PR'], 3)} [{num(IC[nombre_u]['AUC-PR'][0], 3)}; {num(IC[nombre_u]['AUC-PR'][1], 3)}], frente a 0,5 y {num(P['Dummy (prior)']['AUC-PR'], 3)} del Dummy. Con el umbral de {num(umbral, 2)}, elegido en el pliegue de validación más reciente del entrenamiento, la precisión es {num(P[nombre_u]['Precision'], 3)} [{num(IC[nombre_u]['Precision'][0], 3)}; {num(IC[nombre_u]['Precision'][1], 3)}], el recall {num(P[nombre_u]['Recall'], 3)} [{num(IC[nombre_u]['Recall'][0], 3)}; {num(IC[nombre_u]['Recall'][1], 3)}] y el F1 {num(P[nombre_u]['F1'], 3)} [{num(IC[nombre_u]['F1'][0], 3)}; {num(IC[nombre_u]['F1'][1], 3)}]; con umbral 0,5, el recall es {num(P[nombre_05]['Recall'], 3)} y el F1 {num(P[nombre_05]['F1'], 3)}. Las probabilidades tienen una media de {num(prob_te.mean(), 3)} frente a una tasa observada de {num(prev_te, 3)} (BSS = {num(bss, 3)}).
6. **Accuracy engañoso.** El Dummy alcanza un accuracy de {num(acc_d, 3)} en prueba sin identificar a ningún cliente; el modelo obtiene {num(acc_m, 3)} con umbral 0,5 y {num(acc_u, 3)} con el umbral elegido. El accuracy refleja la proporción de la clase mayoritaria y no sirve para evaluar este problema.
7. **Respuesta a la pregunta de investigación.** Con información previa a la llamada y respetando el orden temporal, un modelo lineal discrimina de forma moderada en campañas posteriores (AUC de {num(P[nombre_u]['AUC'], 3)} en prueba, {enumerar([f"{num(r['AUC'], 3)} en {a}" for a, r in por_anio_te.iterrows()], codigo=False)} dentro de cada año de prueba, y {num(grid.best_score_, 3)} en la validación temporal), por encima de la línea base trivial, pero su desempeño depende del periodo y sus probabilidades no se transfieren sin ajuste cuando cambia la prevalencia (media predicha de {num(prob_te.mean(), 3)} frente a una tasa de {num(prev_te, 3)} en prueba).

### 5.2 Limitaciones

- **Validación:** la prueba es un único corte temporal ({periodo_te.iloc[0]} a {periodo_te.iloc[-1]}); otro corte podría dar un resultado distinto, porque el desempeño cambia entre periodos. El corte cae dentro de {periodo_corte}, único mes presente en entrenamiento y prueba.
- **Entrenamiento dominado por 2008:** el {pct((anio_tr == 2008).mean())} del entrenamiento es de 2008, un periodo con pocas suscripciones y condiciones económicas distintas a las de la prueba; por eso las probabilidades quedan desplazadas en prueba, y el umbral se eligió con el bloque de validación más reciente, que solo llega hasta {periodo_tr.iloc[-1]} y es un único bloque.
- **Tiempo reconstruido:** el año y los días de contacto se reconstruyeron a partir del orden documentado de las filas; no hay una fecha explícita, así que no se hizo el análisis temporal completo (descomposición, estacionariedad, ACF de la serie).
- **Entidades:** no hay identificador de cliente; un mismo cliente puede aparecer varias veces y no se pudo usar `GroupKFold`, de modo que el tamaño efectivo de la muestra puede ser menor que el número de filas.
- **Disponibilidad de variables:** `campaign` puede depender del resultado (es razonable suponer que la campaña deja de llamar a quien ya suscribió) y los indicadores macroeconómicos podrían no estar publicados el día de la llamada; ambos quedan en observación.
- **Modelo:** la regresión logística no captura no linealidades (por ejemplo, la edad) ni interacciones; los coeficientes son asociaciones del periodo de entrenamiento y no efectos causales. Los meses con pocos registros en entrenamiento ({enumerar(infrec_f.get('month', []), codigo=False)}) quedaron agrupados como categoría infrecuente, y los meses no vistos ({enumerar(meses_no_vistos_te, codigo=False) or 'ninguno'}) se codifican igual en prueba, lo que distorsiona sus predicciones.
- **Sesgos de muestreo y representatividad:** un solo banco portugués, clientes elegidos por el banco para sus campañas y un periodo de crisis financiera con el {num(por_anio.loc[2008, '% de registros'], 1)} % de los registros en 2008. Los resultados no se pueden generalizar a otras poblaciones o épocas sin validación.
- **Ética:** el modelo usa edad, estado civil, ocupación y situación crediticia; priorizar llamadas con base en ellos puede concentrar la presión comercial en grupos específicos. Un uso real exigiría auditar el efecto por grupos y limitar el uso de los datos financieros al propósito declarado.
"""))


### 5.1 Conclusiones

1. **Base de datos.** *Bank Marketing* (UCI, CC BY 4.0) cumple el tamaño exigido (41 188 registros, 20 predictoras) y plantea un problema de clasificación desbalanceado (11,3 % de positivos). Su calidad es buena en forma (sin errores de escritura ni nulos explícitos), pero tiene faltantes codificados como `unknown` en el 26,0 % de los registros, que no son MCAR y son informativos, 12 duplicados exactos y un código especial (`pdays = 999`) inconsistente con `previous` en 4 110 registros.
2. **Partición.** Las filas están ordenadas en el tiempo, así que la partición es cronológica: el entrenamiento va de 2008-05 a 2009-05 y la prueba de 2009-05 a 2010-11. La prevalencia pasa del 6,4 % en entrenamiento al 30,8 % en prueba, un cambio que el modelo no puede observar durante el ajuste.
3. **Fuga.** `duration` es una fuga evidente: se conoce solo al terminar la llamada y sube el AUC de validación temporal de 0,540 a 0,856. Se excluyó. Dentro de entrenamiento, una validación aleatoria habría dado AUC de 0,670 con el instrumento de auditoría y un modelo que solo conoce el periodo, 0,660, frente a 0,540 con validación temporal: esa **fuga temporal** justifica la partición y la validación cronológicas.
4. **EDA.** Sin `duration`, la mayor información mutua con `y` es la de `euribor3m`, `cons.conf.idx`, `cons.price.idx` y `month`, que describen el contexto del periodo; el historial con el banco y el canal aportan menos, y los atributos del cliente, todavía menos. Las macro son redundantes entre sí (VIF inicial de hasta 46,2), por lo que se retiraron `euribor3m` y `emp.var.rate`. Las subpoblaciones encontradas corresponden a regímenes económicos y al historial de contacto, no a perfiles sociodemográficos.
5. **Modelo base.** La regresión logística (C = 100) logra en prueba AUC de 0,677 [0,640; 0,704] y AUC-PR de 0,489 [0,448; 0,531], frente a 0,5 y 0,308 del Dummy. Con el umbral de 0,60, elegido en el pliegue de validación más reciente del entrenamiento, la precisión es 0,601 [0,558; 0,647], el recall 0,281 [0,231; 0,337] y el F1 0,383 [0,332; 0,433]; con umbral 0,5, el recall es 0,336 y el F1 0,428. Las probabilidades tienen una media de 0,243 frente a una tasa observada de 0,308 (BSS = 0,212).
6. **Accuracy engañoso.** El Dummy alcanza un accuracy de 0,692 en prueba sin identificar a ningún cliente; el modelo obtiene 0,724 con umbral 0,5 y 0,721 con el umbral elegido. El accuracy refleja la proporción de la clase mayoritaria y no sirve para evaluar este problema.
7. **Respuesta a la pregunta de investigación.** Con información previa a la llamada y respetando el orden temporal, un modelo lineal discrimina de forma moderada en campañas posteriores (AUC de 0,677 en prueba, 0,593 en 2009 y 0,585 en 2010 dentro de cada año de prueba, y 0,542 en la validación temporal), por encima de la línea base trivial, pero su desempeño depende del periodo y sus probabilidades no se transfieren sin ajuste cuando cambia la prevalencia (media predicha de 0,243 frente a una tasa de 0,308 en prueba).

### 5.2 Limitaciones

- **Validación:** la prueba es un único corte temporal (2009-05 a 2010-11); otro corte podría dar un resultado distinto, porque el desempeño cambia entre periodos. El corte cae dentro de 2009-05, único mes presente en entrenamiento y prueba.
- **Entrenamiento dominado por 2008:** el 84,0 % del entrenamiento es de 2008, un periodo con pocas suscripciones y condiciones económicas distintas a las de la prueba; por eso las probabilidades quedan desplazadas en prueba, y el umbral se eligió con el bloque de validación más reciente, que solo llega hasta 2009-05 y es un único bloque.
- **Tiempo reconstruido:** el año y los días de contacto se reconstruyeron a partir del orden documentado de las filas; no hay una fecha explícita, así que no se hizo el análisis temporal completo (descomposición, estacionariedad, ACF de la serie).
- **Entidades:** no hay identificador de cliente; un mismo cliente puede aparecer varias veces y no se pudo usar `GroupKFold`, de modo que el tamaño efectivo de la muestra puede ser menor que el número de filas.
- **Disponibilidad de variables:** `campaign` puede depender del resultado (es razonable suponer que la campaña deja de llamar a quien ya suscribió) y los indicadores macroeconómicos podrían no estar publicados el día de la llamada; ambos quedan en observación.
- **Modelo:** la regresión logística no captura no linealidades (por ejemplo, la edad) ni interacciones; los coeficientes son asociaciones del periodo de entrenamiento y no efectos causales. Los meses con pocos registros en entrenamiento (dec y oct) quedaron agrupados como categoría infrecuente, y los meses no vistos (sep) se codifican igual en prueba, lo que distorsiona sus predicciones.
- **Sesgos de muestreo y representatividad:** un solo banco portugués, clientes elegidos por el banco para sus campañas y un periodo de crisis financiera con el 67,2 % de los registros en 2008. Los resultados no se pueden generalizar a otras poblaciones o épocas sin validación.
- **Ética:** el modelo usa edad, estado civil, ocupación y situación crediticia; priorizar llamadas con base en ellos puede concentrar la presión comercial en grupos específicos. Un uso real exigiría auditar el efecto por grupos y limitar el uso de los datos financieros al propósito declarado.
